# Tol TTS — Vocal Separation with Demucs (GPU)

This notebook strips background music from ScriptureEarth audio Bible MP3s using Meta's Demucs model on a free Colab T4 GPU.

**Steps:**
1. Mount Google Drive
2. Install Demucs
3. Run vocal separation on all MP3s
4. Results saved to `CleanVocals/` folder on Drive

**Before running:** Make sure your runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU)

In [ ]:
#@title 1. Mount Google Drive & Configure Paths
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# EDIT THIS PATH to match your Google Drive folder
# ============================================================
INPUT_FOLDER = '/content/drive/MyDrive/ScriptureEarth_NT_Audio'  #@param {type:"string"}

# Output folder (will be created automatically)
OUTPUT_FOLDER = '/content/drive/MyDrive/ScriptureEarth_NT_Audio_CleanVocals'  #@param {type:"string"}

import os
assert os.path.isdir(INPUT_FOLDER), f'INPUT_FOLDER not found: {INPUT_FOLDER}\nCheck the path and try again.'
mp3s = sorted([f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith('.mp3')])
print(f'Found {len(mp3s)} MP3 files in {INPUT_FOLDER}')
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
already = set(os.listdir(OUTPUT_FOLDER))
remaining = [f for f in mp3s if f not in already]
print(f'Already separated: {len(mp3s) - len(remaining)}')
print(f'Remaining: {len(remaining)}')

In [ ]:
#@title 2. Install Demucs + Verify GPU
!pip install -q demucs

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f'GPU: {gpu}')
assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → T4 GPU'

In [ ]:
#@title 3. Run Vocal Separation (all files)
import subprocess, shutil, time, os

TEMP_OUT = '/content/demucs_output'

mp3s = sorted([f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith('.mp3')])
already = set(os.listdir(OUTPUT_FOLDER))
remaining = [f for f in mp3s if f not in already]

total = len(remaining)
print(f'Processing {total} files on GPU...\n')
t0 = time.time()
ok = 0
failed = []

BATCH = 5
for i in range(0, total, BATCH):
    batch = remaining[i:i+BATCH]
    paths = [os.path.join(INPUT_FOLDER, f) for f in batch]

    if os.path.isdir(TEMP_OUT):
        shutil.rmtree(TEMP_OUT)

    cmd = ['python', '-m', 'demucs',
           '--two-stems', 'vocals',
           '-n', 'htdemucs',
           '-d', 'cuda',
           '--mp3',
           '-o', TEMP_OUT] + paths

    result = subprocess.run(cmd, capture_output=True, text=True)

    for f in batch:
        stem = os.path.splitext(f)[0]
        vocal = os.path.join(TEMP_OUT, 'htdemucs', stem, 'vocals.mp3')
        if os.path.exists(vocal):
            shutil.copy2(vocal, os.path.join(OUTPUT_FOLDER, f))
            ok += 1
        else:
            failed.append(f)

    done = ok + len(failed)
    elapsed = time.time() - t0
    rate = elapsed / max(done, 1)
    eta = rate * (total - done) / 60
    print(f'  [{done}/{total}] ({done/total*100:.0f}%) ok={ok} fail={len(failed)} | ETA ~{eta:.0f}m')

elapsed = time.time() - t0
print(f'\n{"="*50}')
print(f'DONE — {ok} separated, {len(failed)} failed')
print(f'Time: {elapsed/60:.1f} minutes')
print(f'Output: {OUTPUT_FOLDER}')
if failed:
    print(f'Failed files: {failed}')

In [ ]:
#@title 4. Verify Results
import os
clean = sorted(os.listdir(OUTPUT_FOLDER))
print(f'Clean vocal files: {len(clean)} / {len(mp3s)}')
total_mb = sum(os.path.getsize(os.path.join(OUTPUT_FOLDER, f)) for f in clean) / 1e6
print(f'Total size: {total_mb:.0f} MB')
print(f'\nReady to download from: {OUTPUT_FOLDER}')